# Phase 2: Baseline Multilingual G2P Model (Transformer)

This notebook implements a **Transformer-based Sequence-to-Sequence** model for **Grapheme-to-Phoneme (G2P)** conversion across Hindi, Gujarati, and Marathi.

**Pipeline:**
1. Load & tokenize the multilingual G2P dataset from Phase 1
2. Build a lightweight Transformer Encoder-Decoder using TensorFlow/Keras
3. Train using Colab GPU
4. Evaluate using Phoneme Error Rate (PER) & Word Error Rate (WER)

## 1. Setup & Imports

In [ ]:
import tensorflow as tf
import numpy as np
import os
import json
import random
from collections import Counter

# For PER calculation
!pip install editdistance -q
import editdistance

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/drive')

# Set the path to your dataset on Google Drive
DATA_PATH = '/content/drive/MyDrive/multilingual_g2p_dataset.txt'  # <-- EDIT THIS PATH

print(f'Dataset path: {DATA_PATH}')
print(f'File exists: {os.path.exists(DATA_PATH)}')

## 2. Data Loading & Tokenization

In [ ]:
def load_dataset(file_path):
    src_sentences = []
    tgt_sentences = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                continue

            src_str = parts[0].strip()
            tgt_str = parts[1].strip()

            if src_str.startswith('<') and '>' in src_str:
                end_idx = src_str.index('>')
                lang_tag = src_str[:end_idx+1]
                word = src_str[end_idx+1:].strip()
            else:
                lang_tag = '<UNK>'
                word = src_str

            src_seq = [lang_tag] + list(word)
            tgt_seq = tgt_str.split()

            src_sentences.append(src_seq)
            tgt_sentences.append(tgt_seq)

    return src_sentences, tgt_sentences

src_sentences, tgt_sentences = load_dataset(DATA_PATH)
print(f"Total samples: {len(src_sentences)}")
print(f"Example source: {src_sentences[0]}")
print(f"Example target: {tgt_sentences[0]}")

In [ ]:
class Tokenizer:
    def __init__(self):
        self.pad_token = '<pad>'
        self.unk_token = '<unk>'
        self.sos_token = '<sos>'
        self.eos_token = '<eos>'
        self.s2i = {self.pad_token: 0, self.unk_token: 1, self.sos_token: 2, self.eos_token: 3}
        self.i2s = {0: self.pad_token, 1: self.unk_token, 2: self.sos_token, 3: self.eos_token}
        self.vocab_size = 4

    def fit(self, sentences):
        for sentence in sentences:
            for token in sentence:
                if token not in self.s2i:
                    self.s2i[token] = self.vocab_size
                    self.i2s[self.vocab_size] = token
                    self.vocab_size += 1

    def encode(self, sentence, add_special=True):
        encoded = [self.s2i.get(tok, self.s2i[self.unk_token]) for tok in sentence]
        if add_special:
            encoded = [self.s2i[self.sos_token]] + encoded + [self.s2i[self.eos_token]]
        return encoded

    def decode(self, indices, remove_special=True):
        special = {self.pad_token, self.sos_token, self.eos_token}
        result = []
        for idx in indices:
            tok = self.i2s.get(idx, self.unk_token)
            if remove_special and tok in special:
                continue
            result.append(tok)
        return result

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'s2i': self.s2i, 'i2s': {int(k): v for k, v in self.i2s.items()}, 'vocab_size': self.vocab_size}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        tok.s2i = data['s2i']
        tok.i2s = {int(k): v for k, v in data['i2s'].items()}
        tok.vocab_size = data['vocab_size']
        return tok

In [ ]:
# ---- Train/Val/Test Split (80/10/10) ----
random.seed(42)
indices = list(range(len(src_sentences)))
random.shuffle(indices)

train_size = int(len(indices) * 0.8)
val_size = int(len(indices) * 0.1)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

train_src = [src_sentences[i] for i in train_idx]
train_tgt = [tgt_sentences[i] for i in train_idx]
val_src   = [src_sentences[i] for i in val_idx]
val_tgt   = [tgt_sentences[i] for i in val_idx]
test_src  = [src_sentences[i] for i in test_idx]
test_tgt  = [tgt_sentences[i] for i in test_idx]

print(f"Train: {len(train_src)} | Val: {len(val_src)} | Test: {len(test_src)}")

src_tokenizer = Tokenizer()
src_tokenizer.fit(train_src)
tgt_tokenizer = Tokenizer()
tgt_tokenizer.fit(train_tgt)

print(f"Source vocab size: {src_tokenizer.vocab_size}")
print(f"Target vocab size: {tgt_tokenizer.vocab_size}")

In [ ]:
# ---- Encode & Pad ----
def encode_and_pad(src_sents, tgt_sents, src_tok, tgt_tok):
    src_encoded = [src_tok.encode(s) for s in src_sents]
    tgt_encoded = [tgt_tok.encode(t) for t in tgt_sents]
    src_padded = tf.keras.preprocessing.sequence.pad_sequences(src_encoded, padding='post', value=0)
    tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(tgt_encoded, padding='post', value=0)
    return src_padded, tgt_padded

train_src_enc, train_tgt_enc = encode_and_pad(train_src, train_tgt, src_tokenizer, tgt_tokenizer)
val_src_enc, val_tgt_enc     = encode_and_pad(val_src, val_tgt, src_tokenizer, tgt_tokenizer)
test_src_enc, test_tgt_enc   = encode_and_pad(test_src, test_tgt, src_tokenizer, tgt_tokenizer)

print(f"Train src shape: {train_src_enc.shape}, tgt shape: {train_tgt_enc.shape}")
print(f"Val   src shape: {val_src_enc.shape},   tgt shape: {val_tgt_enc.shape}")
print(f"Test  src shape: {test_src_enc.shape},  tgt shape: {test_tgt_enc.shape}")

In [ ]:
# ---- Create tf.data.Dataset pipelines ----
BATCH_SIZE = 64
BUFFER_SIZE = 20000

def make_dataset(src, tgt, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((src, tgt))
    if shuffle:
        ds = ds.shuffle(BUFFER_SIZE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_src_enc, train_tgt_enc, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_src_enc, val_tgt_enc, BATCH_SIZE)
test_ds  = make_dataset(test_src_enc, test_tgt_enc, BATCH_SIZE)

for src_batch, tgt_batch in train_ds.take(1):
    print(f"Source batch: {src_batch.shape}")
    print(f"Target batch: {tgt_batch.shape}")

## 3. Transformer Model Architecture

All layer `call()` methods use **keyword-only** arguments for `training` and `mask` to be compatible with Keras 3.

In [ ]:
# ---- Positional Encoding ----
def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], dtype=tf.float32)

# ---- Masking Utilities ----
def create_padding_mask(seq):
    mask = tf.cast(tf.math.equal(seq, 0), tf.float32)
    return mask[:, tf.newaxis, tf.newaxis, :]

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask

def create_masks(src, tgt):
    enc_padding_mask = create_padding_mask(src)
    dec_padding_mask = create_padding_mask(src)
    look_ahead_mask = create_look_ahead_mask(tf.shape(tgt)[1])
    dec_target_padding_mask = create_padding_mask(tgt)
    combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)
    return enc_padding_mask, combined_mask, dec_padding_mask

In [ ]:
# ---- Scaled Dot-Product Attention (functional) ----
def scaled_dot_product_attention(q, k, v, mask):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled = matmul_qk / tf.math.sqrt(dk)
    if mask is not None:
        scaled += (mask * -1e9)
    weights = tf.nn.softmax(scaled, axis=-1)
    output = tf.matmul(weights, v)
    return output

# ---- Multi-Head Attention ----
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0
        self.depth = d_model // num_heads
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs, **kwargs):
        # inputs is a dict: {'v': ..., 'k': ..., 'q': ..., 'mask': ...}
        v, k, q, mask = inputs['v'], inputs['k'], inputs['q'], inputs['mask']
        batch_size = tf.shape(q)[0]
        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)
        output = scaled_dot_product_attention(q, k, v, mask)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat = tf.reshape(output, (batch_size, -1, self.d_model))
        return self.dense(concat)

In [ ]:
# ---- Feed-Forward Network ----
def point_wise_ffn(d_model, dff):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ])

# ---- Encoder Layer ----
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_ffn(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, mask = inputs
        attn = self.mha({'v': x, 'k': x, 'q': x, 'mask': mask})
        attn = self.dropout1(attn, training=training)
        out1 = self.ln1(x + attn)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.ln2(out1 + ffn_out)

# ---- Decoder Layer ----
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_ffn(d_model, dff)
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout3 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, enc_output, look_ahead_mask, padding_mask = inputs
        attn1 = self.mha1({'v': x, 'k': x, 'q': x, 'mask': look_ahead_mask})
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.ln1(attn1 + x)
        attn2 = self.mha2({'v': enc_output, 'k': enc_output, 'q': out1, 'mask': padding_mask})
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.ln2(attn2 + out1)
        ffn_out = self.ffn(out2)
        ffn_out = self.dropout3(ffn_out, training=training)
        return self.ln3(ffn_out + out2)

In [ ]:
# ---- Encoder ----
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, max_pos_enc, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(input_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_pos_enc, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, mask = inputs
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer((x, mask), training=training)
        return x

# ---- Decoder ----
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size, max_pos_enc, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(target_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_pos_enc, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, dropout_rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False, **kwargs):
        x, enc_output, look_ahead_mask, padding_mask = inputs
        seq_len = tf.shape(x)[1]
        x = self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)
        for layer in self.dec_layers:
            x = layer((x, enc_output, look_ahead_mask, padding_mask), training=training)
        return x

In [ ]:
# ---- Full Transformer Model ----
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff,
                 input_vocab_size, target_vocab_size,
                 pe_input, pe_target, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                               input_vocab_size, pe_input, dropout_rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                               target_vocab_size, pe_target, dropout_rate)
        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs, training=False, **kwargs):
        src, tgt = inputs
        enc_padding_mask, look_ahead_mask, dec_padding_mask = create_masks(src, tgt)
        enc_output = self.encoder((src, enc_padding_mask), training=training)
        dec_output = self.decoder((tgt, enc_output, look_ahead_mask, dec_padding_mask), training=training)
        final_output = self.final_layer(dec_output)
        return final_output

## 4. Hyperparameters & Model Initialization

In [ ]:
NUM_LAYERS = 3
D_MODEL = 128
NUM_HEADS = 4
DFF = 512
DROPOUT_RATE = 0.1
EPOCHS = 30

MAX_SRC_LEN = train_src_enc.shape[1] + 50
MAX_TGT_LEN = train_tgt_enc.shape[1] + 50

model = Transformer(
    num_layers=NUM_LAYERS, d_model=D_MODEL, num_heads=NUM_HEADS, dff=DFF,
    input_vocab_size=src_tokenizer.vocab_size,
    target_vocab_size=tgt_tokenizer.vocab_size,
    pe_input=MAX_SRC_LEN, pe_target=MAX_TGT_LEN,
    dropout_rate=DROPOUT_RATE
)

print(f"Model created: {NUM_LAYERS} layers, d_model={D_MODEL}, heads={NUM_HEADS}")

In [ ]:
# ---- Learning Rate Schedule ----
class TransformerSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super().__init__()
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

lr_schedule = TransformerSchedule(D_MODEL)
optimizer = tf.keras.optimizers.Adam(lr_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

In [ ]:
# ---- Loss Function (ignore padding) ----
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def accuracy_function(real, pred):
    pred_ids = tf.argmax(pred, axis=2)
    real = tf.cast(real, dtype=pred_ids.dtype)
    match = tf.cast(tf.equal(real, pred_ids), tf.float32)
    mask = tf.cast(tf.math.logical_not(tf.math.equal(real, 0)), tf.float32)
    return tf.reduce_sum(match * mask) / tf.reduce_sum(mask)

## 5. Training Loop

In [ ]:
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.Mean(name='train_accuracy')

@tf.function
def train_step(src, tgt):
    tgt_input = tgt[:, :-1]
    tgt_real  = tgt[:, 1:]
    with tf.GradientTape() as tape:
        predictions = model((src, tgt_input), training=True)
        loss = loss_function(tgt_real, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_loss(loss)
    train_accuracy(accuracy_function(tgt_real, predictions))

@tf.function
def val_step(src, tgt):
    tgt_input = tgt[:, :-1]
    tgt_real  = tgt[:, 1:]
    predictions = model((src, tgt_input), training=False)
    return loss_function(tgt_real, predictions)

In [ ]:
# ---- Main Training Loop ----
best_val_loss = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss.reset_state()
    train_accuracy.reset_state()

    for batch, (src, tgt) in enumerate(train_ds):
        train_step(src, tgt)

    val_losses = []
    for src, tgt in val_ds:
        v_loss = val_step(src, tgt)
        val_losses.append(v_loss.numpy())
    avg_val_loss = np.mean(val_losses)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss.result():.4f} | "
          f"Train Acc: {train_accuracy.result():.4f} | "
          f"Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        model.save_weights('/content/drive/MyDrive/best_g2p_transformer.weights.h5')
        print("  -> Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("\nTraining complete!")

## 6. Evaluation (PER & WER)

In [ ]:
def greedy_decode(model, src, max_len=100):
    sos_id = tgt_tokenizer.s2i[tgt_tokenizer.sos_token]
    eos_id = tgt_tokenizer.s2i[tgt_tokenizer.eos_token]
    decoder_input = tf.expand_dims([sos_id], 0)

    for _ in range(max_len):
        predictions = model((src, decoder_input), training=False)
        next_token = tf.argmax(predictions[:, -1, :], axis=-1)
        next_token = tf.expand_dims(next_token, 0)
        decoder_input = tf.concat([decoder_input, next_token], axis=-1)
        if next_token.numpy()[0][0] == eos_id:
            break

    return decoder_input.numpy()[0].tolist()

In [ ]:
# Load best weights
model.load_weights('/content/drive/MyDrive/best_g2p_transformer.weights.h5')
print("Loaded best model weights.")

In [ ]:
# ---- Calculate PER and WER on Test Set ----
total_per = 0.0
total_wer = 0.0
num_samples = 0

for i in range(len(test_src_enc)):
    src_input = tf.expand_dims(test_src_enc[i], 0)
    predicted_ids = greedy_decode(model, src_input, max_len=50)

    pred_phonemes = tgt_tokenizer.decode(predicted_ids, remove_special=True)
    true_phonemes = tgt_tokenizer.decode(test_tgt_enc[i].tolist(), remove_special=True)

    per = editdistance.eval(pred_phonemes, true_phonemes) / max(len(true_phonemes), 1)
    total_per += per

    wer = 0.0 if pred_phonemes == true_phonemes else 1.0
    total_wer += wer
    num_samples += 1

    if i < 10:
        src_decoded = src_tokenizer.decode(test_src_enc[i].tolist(), remove_special=True)
        print(f"Input:     {''.join(src_decoded)}")
        print(f"Predicted: {' '.join(pred_phonemes)}")
        print(f"Expected:  {' '.join(true_phonemes)}")
        print(f"PER: {per:.4f}")
        print('-' * 50)

avg_per = total_per / num_samples
avg_wer = total_wer / num_samples

print(f"\n{'='*50}")
print(f"BASELINE RESULTS ON TEST SET ({num_samples} samples)")
print(f"{'='*50}")
print(f"Phoneme Error Rate (PER): {avg_per:.4f} ({avg_per*100:.2f}%)")
print(f"Word Error Rate   (WER):  {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"{'='*50}")

## 7. Save Tokenizers

In [ ]:
src_tokenizer.save('/content/drive/MyDrive/src_tokenizer.json')
tgt_tokenizer.save('/content/drive/MyDrive/tgt_tokenizer.json')
print("Tokenizers saved to Google Drive.")
print("Model weights saved as best_g2p_transformer.weights.h5 on Google Drive.")